# Second Encoder

The primary change here will be the separation criteria that we will use, rather than just ionization mode we will also differentiate by instrument type. 

In [1]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# from fcd_torch import FCD
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

from fcd_torch import FCD
import rdkit

### Data Upload and basic processing

In [2]:
# The 5/20 dataset with rat based toxicity data
df3 = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/MIT_LL_data3.csv")
print(df3.shape)
df3.head()

(4001, 16)


,SMILES_spectra,CAS,Molecular_Formula,Total_Exact_Mass,Precursor_m/z,Precursor_Type,Spectrum,Ionization_Mode,Instrument_Type,Instrument_Name,Collision_Energy,SMILES_tox_vals,Response_Modifier,Response,Response_Unit,Group
0,'C#CCN(C)Cc1ccccc1','555-57-7','C11H13N','159.104799416','160.1121','[M+H]+','63.0228:0.177223 65.0386:5.629055 68.0495:0.4...,'positive','LC-ESI-QFT','Q Exactive Orbitrap Thermo Scientific','90 (nominal)','C#CCN(C)Cc1ccccc1','',273.642508,'mg/kg','Q-Orbitrap-positive'
1,'C#CCN(C)Cc1ccccc1','555-57-7','C11H13N','159.104799416','160.1121','[M+H]+','63.0228:0.125979 65.0386:2.113734 68.0495:0.6...,'positive','LC-ESI-QFT','Q Exactive Orbitrap Thermo Scientific','75 (nominal)','C#CCN(C)Cc1ccccc1','',273.642508,'mg/kg','Q-Orbitrap-positive'
2,'C#CCN(C)Cc1ccccc1','555-57-7','C11H13N','159.104799416','160.1121','[M+H]+','56.0496:0.115017 65.0386:0.970445 68.0495:1.0...,'positive','LC-ESI-QFT','Q Exactive Orbitrap Thermo Scientific','60 (nominal)','C#CCN(C)Cc1ccccc1','',273.642508,'mg/kg','Q-Orbitrap-positive'
3,'C#CCN(C)Cc1ccccc1','555-57-7','C11H13N','159.104799416','160.1121','[M+H]+','51.0229:0.102992 56.0495:0.143820 65.0386:0.6...,'positive','LC-ESI-QFT','Q Exactive Orbitrap Thermo Scientific','45 (nominal)','C#CCN(C)Cc1ccccc1','',273.642508,'mg/kg','Q-Orbitrap-positive'
4,'C#CCN(C)Cc1ccccc1','555-57-7','C11H13N','159.104799416','160.1121','[M+H]+','56.0496:0.482623 65.0385:0.377829 68.0495:2.5...,'positive','LC-ESI-QFT','Q Exactive Orbitrap Thermo Scientific','30 (nominal)','C#CCN(C)Cc1ccccc1','',273.642508,'mg/kg','Q-Orbitrap-positive'


In [3]:
# Uniformity of ionization model labels
print(df3["Ionization_Mode"].unique())
df3["Ionization_Mode"] = df3["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df3["Ionization_Mode"].unique())

# Remove the N/A values in Ionization_Mode
df3 = df3[df3["Ionization_Mode"] != "'N/A'"]
print(df3["Ionization_Mode"].unique())

# Removing the '' from the SMILES
# Remove single quotes from all string columns in df3
df3 = df3.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)
#df3["SMILES_spectra"] = df3["SMILES_spectra"].str.replace("'", "")
df3.head()

["'positive'" "'negative'" "'Positive'" "'NaN'"]
["'positive'" "'negative'" "'NaN'"]
["'positive'" "'negative'" "'NaN'"]


,SMILES_spectra,CAS,Molecular_Formula,Total_Exact_Mass,Precursor_m/z,Precursor_Type,Spectrum,Ionization_Mode,Instrument_Type,Instrument_Name,Collision_Energy,SMILES_tox_vals,Response_Modifier,Response,Response_Unit,Group
0,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,63.0228:0.177223 65.0386:5.629055 68.0495:0.45...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,90 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive
1,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,63.0228:0.125979 65.0386:2.113734 68.0495:0.68...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,75 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive
2,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,56.0496:0.115017 65.0386:0.970445 68.0495:1.03...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,60 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive
3,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,51.0229:0.102992 56.0495:0.143820 65.0386:0.67...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,45 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive
4,C#CCN(C)Cc1ccccc1,555-57-7,C11H13N,159.104799416,160.1121,[M+H]+,56.0496:0.482623 65.0385:0.377829 68.0495:2.59...,positive,LC-ESI-QFT,Q Exactive Orbitrap Thermo Scientific,30 (nominal),C#CCN(C)Cc1ccccc1,,273.642508,mg/kg,Q-Orbitrap-positive


In [7]:
print(df3['Group'].nunique()) # 13
# 13 groups means a 13 way split
print(df3['Group'].unique())
# Now we want counts of each group
print(df3['Group'].value_counts())

13
['Q-Orbitrap-positive' 'Q-Orbitrap-negative' 'Q-TOF-positive'
 'LTQ-Orbitrap-positive' 'QQQ-positive' 'Q-TOF-negative'
 'LTQ-Orbitrap-negative' 'QQQ-negative' 'Other-positive' 'LTQ-negative'
 'QQQ-nan' 'LTQ-positive' 'Other-negative']
Group
Q-Orbitrap-positive      1307
Q-Orbitrap-negative       756
Q-TOF-positive            736
LTQ-Orbitrap-positive     481
QQQ-positive              253
Q-TOF-negative            188
QQQ-negative               85
Other-positive             71
LTQ-Orbitrap-negative      63
LTQ-positive               19
QQQ-nan                    18
Other-negative             13
LTQ-negative               11
Name: count, dtype: int64


Notice that in this case we needn't do a positive negative split since that split is aready included inthe group differentiation. 